## Generate data

In [ ]:
import numpy as np
import rioxarray  # noqa: F401
import xarray as xra

NPIXELS_Y = 100
NPIXELS_X = 100
MIN_X = -180
MAX_X = 180
MIN_Y = -90
MAX_Y = 90
RES_X = (MAX_X - MIN_X) / NPIXELS_X
RES_Y = (MAX_Y - MIN_Y) / NPIXELS_Y

# Coordinates at pixel centers
y_coords = np.linspace(MAX_Y - RES_Y / 2, MIN_Y + RES_Y / 2, NPIXELS_Y)
x_coords = np.linspace(MIN_X + RES_X / 2, MAX_X - RES_X / 2, NPIXELS_X)
x_grid, y_grid = np.meshgrid(x_coords, y_coords)

data = ((x_grid - x_grid.min()) + (y_grid - y_grid.min())) / 2  # Diagonal gradient
data = data / data.max()  # Normalize to 0-1

da = xra.DataArray(
    data,
    dims=["y", "x"],
    coords={
        "y": y_coords,
        "x": x_coords,
    },
)
da.rio.write_crs("EPSG:4326", inplace=True)
da

In [ ]:
import cartopy.crs as ccrs
import matplotlib.pyplot as plt

map_proj = ccrs.PlateCarree()

fig = plt.figure(figsize=(10, 6))
ax = fig.add_subplot(1, 1, 1, projection=map_proj)

da.plot.pcolormesh(ax=ax, transform=map_proj)

ax.coastlines()
ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False)
ax.set_extent([-180, 180, -90, 90], crs=map_proj)  # Optional: set specific map extent

plt.show()

## Render data

In [ ]:
from jupyter_xarray_tiler.titiler import add_data_array

url = await add_data_array(da, rescale=(0, 1))

In [ ]:
url

In [ ]:
import leafmap

m = leafmap.Map(center=[41.321482, -72.932739], zoom=10)
m

In [ ]:
m.add_tile_layer(
    url=url,
    name="Mock data",
    attribution="¯\\_(ツ)_/¯",
)

In [ ]:
da.spatial_ref